In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box
import geobr
import time
import sys, os

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Importa métodos e/ou funções 
from spark_utils import get_spark_session, write_data_csv, convert_nc_to_spark_dataframe

In [ ]:
# Cria uma conexão Spark 
spark = get_spark_session("Municipio_Shape")

In [ ]:

def criar_grade_copernicus(min_lon, min_lat, max_lon, max_lat, step=0.25):
    """
    Constrói os polígonos quadrados correspondentes às células do Copernicus ERA5/ERA5-Land.
    step=0.25 para resolução de ~27.5 km x 27.5 km
    (Altere step=0.1 se estiver usando a malha de alta resolução do ERA5-Land ~9 km).
    """
    print(f"--> Gerando grade sintética do Copernicus (Resolução: {step}°)...")
    grid_cells = []
    half = step / 2.0
    
    # Criar eixos de coordenadas centralizados
    lons = np.arange(min_lon, max_lon + step, step)
    lats = np.arange(min_lat, max_lat + step, step)
    
    for lat in lats:
        for lon in lons:
            # Polígono quadrado em torno da coordenada central
            cell_poly = box(lon - half, lat - half, lon + half, lat + half)
            grid_cells.append({
                'id_geo': f"GRID_{lat:.2f}_{lon:.2f}",
                'latitude_centro': round(float(lat), 4),
                'longitude_centro': round(float(lon), 4),
                'geometry': cell_poly
            })
            
    gdf_grid = gpd.GeoDataFrame(grid_cells, crs="EPSG:4326")
    print(f"    Total de células geradas na grade: {len(gdf_grid):,}")
    return gdf_grid


def calcular_pesos_sobreposicao(gdf_grid, gdf_municipios, crs_metrico="EPSG:5880"):
    """
    Calcula a área de sobreposição entre a grade climática e os municípios em m².
    EPSG:5880 = SIRGAS 2000 / Brasil Polyconic (Ideal para medições de área no Brasil).
    """
    print("--> Reprojetando geometrias para sistema métrico plano (m²)...")
    grid_proj = gdf_grid.to_crs(crs_metrico)
    mun_proj = gdf_municipios.to_crs(crs_metrico)
    
    # Área total real de cada município do IBGE em m²
    print("--> Calculando áreas totais dos municípios...")
    mun_proj['area_municipio_m2'] = mun_proj.geometry.area
    
    print("--> Executando intersecção geométrica (Overlay Espacial)... isso pode levar de 1 a 3 minutos.")
    start_overlay = time.time()
    
    # Intersecção entre a malha e os municípios
    intersection = gpd.overlay(grid_proj, mun_proj, how='intersection')
    
    end_overlay = time.time()
    print(f"    Intersecção concluída em {end_overlay - start_overlay:.2f} segundos.")
    
    # Área do pedaço (fração) que sobrepõe
    intersection['area_interseccao_m2'] = intersection.geometry.area
    
    print("--> Normalizando os fatores de peso por área...")
    # FATOR DE PESO: Quanto dessa célula representa a área TOTAL do município
    # A soma de 'fator_peso_municipio' para um mesmo município será exatamente 1.0 (100%)
    intersection['fator_peso_municipio'] = (
        intersection['area_interseccao_m2'] / intersection['area_municipio_m2']
    )
    
    # Arredondar para evitar dízimas no Spark
    intersection['fator_peso_municipio'] = intersection['fator_peso_municipio'].round(6)
    
    # Filtrar pequenas ruínas de borda insignificantes (< 0.01% da área do município)
    intersection = intersection[intersection['fator_peso_municipio'] > 0.0001].copy()
    
    # Seleção final de colunas ajustadas para o Modelo Dimensional
    colunas_finais = [
        'id_geo',
        'latitude_centro',
        'longitude_centro',
        'code_muni',           # Código IBGE (ex: 3550308)
        'name_muni',           # Nome do Município (ex: São Paulo)
        'abbrev_state',        # UF (ex: SP)
        'area_municipio_m2',
        'area_interseccao_m2',
        'fator_peso_municipio'
    ]
    
    df_result = pd.DataFrame(intersection[colunas_finais])
    return df_result



In [ ]:

tempo_inicio = time.time()
print("=== INICIANDO CONSTRUÇÃO DA LOOKUP TABLE ESPACIAL (IBGE x COPERNICUS) ===")

# 1. Baixar mapa dos municípios do IBGE via biblioteca geobr
print("\n[Passo 1/4] Baixando a malha oficial de municípios do IBGE (Brasil inteiro)...")
# Para testar apenas um estado primeiro, você pode trocar "all" por "SP" ou "MG"
gdf_ibge = geobr.read_municipality(code_muni="all", year=2022)

# 2. Obter limites territoriais do Brasil para recortar a grade do Copernicus
bounds = gdf_ibge.total_bounds  # [min_lon, min_lat, max_lon, max_lat]

# 3. Construir a grade estática do Copernicus
print("\n[Passo 2/4] Construindo malha do Copernicus...")
gdf_copernicus = criar_grade_copernicus(
    min_lon=bounds[0] - 0.25,
    min_lat=bounds[1] - 0.25,
    max_lon=bounds[2] + 0.25,
    max_lat=bounds[3] + 0.25,
    step=0.25  # Alterar para 0.1 se seus dados Copernicus forem na resolução de 9 km
)

# 4. Processar a sobreposição de áreas (Overlay)
print("\n[Passo 3/4] Processando Overlay Espacial e Ponderação de Áreas...")
df_lookup = calcular_pesos_sobreposicao(gdf_copernicus, gdf_ibge)

# 5. Salvar arquivo Parquet
print("\n[Passo 4/4] Exportando tabela final...")
nome_arquivo = "d_lookup_grid_municipio_brasil.parquet"
df_lookup.to_parquet(nome_arquivo, index=False)

tempo_total = time.time() - tempo_inicio
print(f"\n✅ CONCLUÍDO COM SUCESSO EM {tempo_total/60:.2f} MINUTOS!")
print(f"--> Tabela gerada: '{nome_arquivo}'")
print(f"--> Total de associações (Célula x Município): {len(df_lookup):,} linhas")

# Exibir amostra dos dados
print("\nAmostra dos dados gerados:")
print(df_lookup.head(10))

In [ ]:
df_grid_municipio = spark.read.parquet(r"C:\Marco Conti\Projetos\mais_einstein\Municipio\d_lookup_grid_municipio_brasil.parquet")

In [ ]:
df_grid_municipio.printSchema()
df_grid_municipio.show(10, False)

In [ ]:
df_temperatura = spark.read.csv(r"C:\Marco Conti\Projetos\Dados\ERA5-temperaturas\arquivos_CSV\ERA5_t2m_2022.csv", sep=",", header=True)
df_temperatura.printSchema()
df_temperatura.shoe(10, False)


In [ ]:
df_grid_municipio.createOrReplaceTempView("d_lookup_grid_municipio")


query_temp_SP = \
    """ SELECT l.code_muni,
            l.name_muni,
            l.abbrev_state AS uf,
            f.data,
            ROUND(SUM(f.temp_media * l.fator_peso_municipio), 2) AS temp_media_municipio,
            ROUND(SUM(f.temp_max   * l.fator_peso_municipio), 2) AS temp_max_municipio,
            ROUND(SUM(f.temp_min   * l.fator_peso_municipio), 2) AS temp_min_municipio
          FROM gold.f_temperatura_diaria f
         INNER JOIN d_lookup_grid_municipio l 
            -- Garante o encaixe perfeito das coordenadas numéricas
            ON  ROUND(f.latitude, 2)  = l.latitude_centro
           AND ROUND(f.longitude, 2) = l.longitude_centro
         GROUP BY 
            l.code_muni, 
            l.name_muni, 
            l.abbrev_state, 
            f.data
    """
